# ST-HopGNN: LTE Forecasting and Baseline Comparisons

Research implementation of recurrent-ESN ST-HopGNN and six comparison models. The dataset is **not included**. Supply authorized CSV files in `data/train/`, `data/validate/`, and `data/test/`. Each CSV must have one time/index column followed by 19 sensor columns, with consistent column order.

The original preprocessing, architecture, training, and evaluation code is retained. Model outputs and checkpoints are written to `results/` and are not part of the repository. Run the cells sequentially.

**Note:** Validation-based checkpoint selection can precede adjacency freezing at epoch 20. The best checkpoint therefore does not necessarily contain the epoch-20 graph.


## Setup and configuration


In [ ]:

# ============================================================
# CELL 1 — ST-HopGNN LTE
# Recurrent ESN + Thresholded Adjacency + Self-Loops
# ============================================================

import os
import json
import time
import random
import platform

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GCNConv

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATA_DIR = os.environ.get("STHOPGNN_DATA_DIR", "./data")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "validate")
TEST_DIR = os.path.join(DATA_DIR, "test")

NUM_SENSORS = 19
WINDOW = 5
HORIZON = 3
WARMUP = 10

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

HID = 128
HOPS = 4
DROPOUT = 0.35

ESN_SPECTRAL_RADIUS = 0.9
ESN_LEAK_RATE = 1.0
ESN_INPUT_SCALE = 0.1
ESN_RESERVOIR_DENSITY = 0.1

ADJ_THRESHOLD = 0.5
GCN_SELF_LOOPS = True

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

BATCH = 128
EPOCHS = 80
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0

FREEZE_EPOCH = 20

# ------------------------------------------------------------
# New experiment directory
# ------------------------------------------------------------

PROJECT_DIR = os.environ.get("STHOPGNN_RESULTS_DIR", "./results")

RUN_NAME = "STHopGNN_LTE_seed42"

RUN_DIR = os.path.join(PROJECT_DIR, RUN_NAME)

os.makedirs(RUN_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    RUN_DIR, "best_model.pth"
)

FINAL_MODEL_PATH = os.path.join(
    RUN_DIR, "final_model.pth"
)

HISTORY_PATH = os.path.join(
    RUN_DIR, "training_history.json"
)

RESULTS_PATH = os.path.join(
    RUN_DIR, "test_results.json"
)

CONFIG_PATH = os.path.join(
    RUN_DIR, "experiment_config.json"
)

SCALER_PATH = os.path.join(
    RUN_DIR, "train_scaler.joblib"
)

# ------------------------------------------------------------
# Save configuration
# ------------------------------------------------------------

CONFIG = {
    "experiment": RUN_NAME,
    "seed": SEED,
    "num_sensors": NUM_SENSORS,
    "window": WINDOW,
    "horizon": HORIZON,
    "warmup": WARMUP,
    "hidden_dim": HID,
    "hops": HOPS,
    "dropout": DROPOUT,
    "esn_spectral_radius": ESN_SPECTRAL_RADIUS,
    "esn_leak_rate": ESN_LEAK_RATE,
    "esn_input_scale": ESN_INPUT_SCALE,
    "esn_reservoir_density": ESN_RESERVOIR_DENSITY,
    "adjacency_threshold": ADJ_THRESHOLD,
    "gcn_self_loops": GCN_SELF_LOOPS,
    "batch_size": BATCH,
    "max_epochs": EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "freeze_after_epochs": FREEZE_EPOCH,
    "loss": "MSE",
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "python_version": platform.python_version()
}

with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=4)

# ------------------------------------------------------------
# Verify data folders
# ------------------------------------------------------------

for name, folder in [
    ("Train", TRAIN_DIR),
    ("Validation", VAL_DIR),
    ("Test", TEST_DIR)
]:

    assert os.path.isdir(folder), (
        f"{name} folder not found: {folder}"
    )

    files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(".csv")
    ])

    print(f"{name}: {len(files)} CSV files available")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n==========================================")
print("ST-HopGNN — THRESHOLDED ADJACENCY")
print("==========================================")

print("Device            :", DEVICE)
print(
    "GPU               :",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "CPU"
)

print("Sensors           :", NUM_SENSORS)
print("Window            :", WINDOW)
print("Horizon           :", HORIZON)
print("ESN reservoir     :", HID)
print("Graph hops        :", HOPS)
print("Adjacency threshold:", ADJ_THRESHOLD)
print("GCN self-loops    :", GCN_SELF_LOOPS)
print("Freeze after      :", FREEZE_EPOCH, "epochs")
print("Run directory     :", RUN_DIR)

print("\nCELL 1 COMPLETED SUCCESSFULLY")


## Data loading and preprocessing


In [ ]:

# ============================================================
# CELL 2 — LTE DATA LOADING AND PREPROCESSING
# ============================================================

class EngineWindowDataset(Dataset):

    def __init__(self, folder, scaler=None):

        files = sorted([
            f for f in os.listdir(folder)
            if f.lower().endswith(".csv")
        ])

        if not files:
            raise ValueError(f"No CSV files found in {folder}")

        all_series = []

        print(f"\nDataset: {folder}")

        for filename in files:

            path = os.path.join(folder, filename)
            df = pd.read_csv(path)

            # First column: time/index
            # Remaining columns: 19 sensor measurements
            arr = df.iloc[:, 1:].to_numpy(
                dtype=np.float32
            )

            assert arr.shape[1] == NUM_SENSORS, (
                f"{filename}: Expected {NUM_SENSORS} sensors, "
                f"found {arr.shape[1]}"
            )

            assert np.isfinite(arr).all(), (
                f"{filename}: NaN or infinity detected"
            )

            assert len(arr) >= WARMUP + WINDOW + HORIZON, (
                f"{filename}: Insufficient timesteps"
            )

            all_series.append(arr)

            print(
                f"  {filename}: "
                f"{len(arr)} timesteps, "
                f"{arr.shape[1]} sensors"
            )

        # Fit MinMaxScaler on training data only
        if scaler is None:

            self.scaler = MinMaxScaler().fit(
                np.vstack(all_series)
            )

        else:
            self.scaler = scaler

        X = []
        Y = []

        for series in all_series:

            series = self.scaler.transform(
                series
            ).astype(np.float32)

            for i in range(
                WARMUP,
                len(series) - WINDOW - HORIZON + 1
            ):

                # Five historical timesteps
                X.append(
                    series[i:i + WINDOW]
                )

                # One target vector, HORIZON steps ahead
                Y.append(
                    series[i + WINDOW + HORIZON - 1]
                )

        self.X = torch.from_numpy(
            np.stack(X).astype(np.float32)
        )

        self.Y = torch.from_numpy(
            np.stack(Y).astype(np.float32)
        )

        print("Total windows:", len(self.X))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

train_ds = EngineWindowDataset(TRAIN_DIR)

val_ds = EngineWindowDataset(
    VAL_DIR,
    scaler=train_ds.scaler
)

test_ds = EngineWindowDataset(
    TEST_DIR,
    scaler=train_ds.scaler
)

# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH,
    shuffle=True,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH,
    shuffle=False,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH,
    shuffle=False,
    pin_memory=(DEVICE.type == "cuda")
)

# ------------------------------------------------------------
# Validate dataset dimensions
# ------------------------------------------------------------

xb, yb = next(iter(train_loader))

print("\n==========================================")
print("LTE DATA VALIDATION SUMMARY")
print("==========================================")

print("Training windows   :", len(train_ds))
print("Validation windows :", len(val_ds))
print("Testing windows    :", len(test_ds))

print("Input batch shape  :", tuple(xb.shape))
print("Target batch shape :", tuple(yb.shape))

assert xb.shape[1:] == (WINDOW, NUM_SENSORS)
assert yb.shape[1:] == (NUM_SENSORS,)

assert torch.isfinite(train_ds.X).all()
assert torch.isfinite(train_ds.Y).all()

print("\nCELL 2 COMPLETED SUCCESSFULLY")


## Recurrent echo state network


In [ ]:

# ============================================================
# CELL 3 — RECURRENT ECHO STATE NETWORK
# Full 5-timestep historical window
# ============================================================

class ESN(nn.Module):

    def __init__(
        self,
        in_dim=1,
        hid=128,
        spectral_radius=0.9,
        leak_rate=1.0,
        input_scale=0.1,
        reservoir_density=0.1
    ):
        super().__init__()

        self.hid = hid
        self.leak_rate = leak_rate

        # Fixed input projection
        Win = torch.randn(in_dim, hid) * input_scale

        # Sparse recurrent reservoir
        Wres = torch.randn(hid, hid)

        mask = (
            torch.rand(hid, hid) < reservoir_density
        ).float()

        Wres = Wres * mask

        # Normalize reservoir spectral radius
        eigenvalues = torch.linalg.eigvals(Wres)
        radius = eigenvalues.abs().max().item()

        if radius < 1e-8:
            raise ValueError(
                "Reservoir spectral radius is zero"
            )

        Wres = Wres * (spectral_radius / radius)

        bias = torch.zeros(hid)

        # Buffers remain fixed but are saved in checkpoints
        self.register_buffer("Win", Win)
        self.register_buffer("Wres", Wres)
        self.register_buffer("bias", bias)

    def forward(self, x):
        """
        Input:  [B, T, N]
        Output: [B, N, HID]
        """

        B, T, N = x.shape

        # Independent reservoir state per sensor
        h = x.new_zeros(B, N, self.hid)

        for t in range(T):

            xt = x[:, t, :].unsqueeze(-1)

            input_term = torch.matmul(
                xt, self.Win
            )

            recurrent_term = torch.matmul(
                h, self.Wres
            )

            candidate = torch.tanh(
                input_term
                + recurrent_term
                + self.bias
            )

            h = (
                (1.0 - self.leak_rate) * h
                + self.leak_rate * candidate
            )

        return h


# ------------------------------------------------------------
# Instantiate ESN
# ------------------------------------------------------------

esn = ESN(
    in_dim=1,
    hid=HID,
    spectral_radius=ESN_SPECTRAL_RADIUS,
    leak_rate=ESN_LEAK_RATE,
    input_scale=ESN_INPUT_SCALE,
    reservoir_density=ESN_RESERVOIR_DENSITY
).to(DEVICE)

# ------------------------------------------------------------
# Test ESN
# ------------------------------------------------------------

xb, yb = next(iter(train_loader))
xb = xb.to(DEVICE)

with torch.no_grad():
    esn_output = esn(xb)

actual_radius = torch.linalg.eigvals(
    esn.Wres.cpu()
).abs().max().item()

# Verify historical sensitivity
xb_modified = xb.clone()
xb_modified[:, 0, :] += 0.1

with torch.no_grad():
    modified_output = esn(xb_modified)

history_difference = (
    modified_output - esn_output
).abs().max().item()

print("\n==========================================")
print("RECURRENT ESN VALIDATION")
print("==========================================")

print("Input shape :", tuple(xb.shape))
print("Output shape:", tuple(esn_output.shape))

print(
    "Trainable ESN parameters:",
    sum(
        p.numel()
        for p in esn.parameters()
        if p.requires_grad
    )
)

print(
    "Reservoir spectral radius:",
    round(actual_radius, 6)
)

print(
    "Historical timestep sensitivity:",
    f"{history_difference:.8f}"
)

assert esn_output.shape == (
    xb.shape[0], NUM_SENSORS, HID
)

assert torch.isfinite(esn_output).all()
assert abs(actual_radius - ESN_SPECTRAL_RADIUS) < 1e-3
assert history_difference > 0

print("\nCELL 3 COMPLETED SUCCESSFULLY")


## Thresholded multi-hop graph aggregation


In [ ]:

# ============================================================
# CELL 4 — THRESHOLDED HOPGNN WITH SELF-LOOPS
# Correct batched graph propagation
# ============================================================

class HopGNN(nn.Module):

    def __init__(self, hid, hops, nodes, drop=0.35):
        super().__init__()

        self.hid = hid
        self.hops = hops
        self.nodes = nodes

        # GCNConv with original self-loop behavior
        self.convs = nn.ModuleList([
            GCNConv(
                hid,
                hid,
                add_self_loops=True
            )
            for _ in range(hops)
        ])

        # Learnable hop embeddings
        self.emb = nn.Parameter(
            torch.randn(hops, hid) * 0.1
        )

        self.drop = nn.Dropout(drop)

    def forward(self, x, adj):
        """
        x   : [B, N, HID]
        adj : [N, N] adjacency logits

        Returns: [B, N, HID]
        """

        B, N, H = x.shape

        assert N == self.nodes
        assert H == self.hid

        # Original thresholding
        adj_prob = torch.sigmoid(adj)

        adj_thresholded = (
            adj_prob * (adj_prob > ADJ_THRESHOLD)
        )

        edge_index = (
            adj_thresholded.nonzero()
            .t()
            .contiguous()
        )

        edge_weight = adj_thresholded[
            edge_index[0],
            edge_index[1]
        ]

        E = edge_index.shape[1]

        # Correct independent graph per batch sample
        offsets = (
            torch.arange(
                B,
                device=x.device
            ) * N
        )

        batched_edges = (
            edge_index.unsqueeze(0)
            + offsets[:, None, None]
        ).permute(1, 0, 2).reshape(
            2, B * E
        ).contiguous()

        batched_weights = edge_weight.repeat(B)

        # Flatten batch and sensor dimensions
        h = x.reshape(B * N, H)

        hop_outputs = []

        for k, conv in enumerate(self.convs):

            h = h + self.emb[k]

            h = conv(
                h,
                batched_edges,
                batched_weights
            )

            h = self.drop(h)

            hop_outputs.append(h)

        # Mean aggregation over four hops
        out = torch.stack(
            hop_outputs,
            dim=0
        ).mean(dim=0)

        return out.reshape(B, N, H)


# ------------------------------------------------------------
# Validate graph propagation and adjacency gradients
# ------------------------------------------------------------

test_gnn = HopGNN(
    hid=HID,
    hops=HOPS,
    nodes=NUM_SENSORS,
    drop=DROPOUT
).to(DEVICE)

test_adj = nn.Parameter(
    torch.rand(
        NUM_SENSORS,
        NUM_SENSORS,
        device=DEVICE
    ) * 0.1
)

test_gnn.train()

test_output = test_gnn(
    esn_output.detach(),
    test_adj
)

test_loss = test_output.square().mean()
test_loss.backward()

adj_gradient = test_adj.grad.abs().sum().item()

active_edges = int(
    (
        torch.sigmoid(test_adj)
        > ADJ_THRESHOLD
    ).sum().item()
)

print("\n==========================================")
print("THRESHOLDED HOPGNN VALIDATION")
print("==========================================")

print("Input shape       :", tuple(esn_output.shape))
print("Output shape      :", tuple(test_output.shape))
print("Active graph edges:", active_edges)
print("Self-loops enabled:", GCN_SELF_LOOPS)
print("Adjacency gradient:", f"{adj_gradient:.10f}")

assert test_output.shape == (
    esn_output.shape[0],
    NUM_SENSORS,
    HID
)

assert torch.isfinite(test_output).all()
assert test_adj.grad is not None
assert adj_gradient > 0

print("\nCELL 4 COMPLETED SUCCESSFULLY")



## ST-HopGNN model


In [ ]:

# ============================================================
# CELL 5 — COMPLETE ST-HopGNN MODEL
# Recurrent ESN + Corrected Multi-Hop Graph Aggregation
# ============================================================

class STHopGNN(nn.Module):

    def __init__(
        self,
        nodes,
        hid=128,
        hops=4
    ):
        super().__init__()

        self.nodes = nodes
        self.hid = hid

        # ----------------------------------------------------
        # RECURRENT ESN
        # ----------------------------------------------------
        self.esn = ESN(
            in_dim=1,
            hid=hid
        )

        # ----------------------------------------------------
        # MULTI-HOP GRAPH NETWORK
        # ----------------------------------------------------
        self.gnn = HopGNN(
            hid=hid,
            hops=hops,
            nodes=nodes,
            drop=DROPOUT
        )

        # ----------------------------------------------------
        # LEARNABLE ADJACENCY
        # Same initialization as original notebook
        # ----------------------------------------------------
        self.adj = nn.Parameter(
            torch.rand(nodes, nodes) * 0.1
        )

        # ----------------------------------------------------
        # DROPOUT AND PREDICTION HEAD
        # ----------------------------------------------------
        self.drop = nn.Dropout(DROPOUT)

        self.recon = nn.Linear(hid, 1)

    def forward(self, x):
        """
        Input:
            x : [B, WINDOW, NUM_SENSORS]

        Output:
            y : [B, NUM_SENSORS]
        """

        # Full temporal window through recurrent ESN
        feats = self.esn(x)

        # Multi-hop graph propagation
        h = self.gnn(
            feats,
            self.adj
        )

        h = self.drop(h)

        # Sensor-wise prediction
        y = self.recon(h).squeeze(-1)

        return y

    def freeze_adj(self):
        """
        Freeze learned adjacency after 20
        completed training epochs.
        """
        self.adj.requires_grad_(False)

        print("Learned adjacency frozen.")

    def get_adjacency(self):
        """
        Return thresholded learned adjacency.
        """
        with torch.no_grad():

            adj = torch.sigmoid(self.adj)

            adj = adj * (
                adj > ADJ_THRESHOLD
            )

        return adj


# ============================================================
# MODEL INITIALIZATION
# ============================================================

model = STHopGNN(
    nodes=NUM_SENSORS,
    hid=HID,
    hops=HOPS
).to(DEVICE)

print("\n==========================================")
print("ST-HopGNN MODEL INITIALIZED")
print("==========================================")

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters     :", total_params)
print("Trainable parameters :", trainable_params)
print("Reservoir size       :", HID)
print("Graph hops           :", HOPS)
print("Adjacency threshold  :", ADJ_THRESHOLD)
print("Adjacency freeze     :", FREEZE_EPOCH)


# ============================================================
# FORWARD PASS VALIDATION
# ============================================================

model.eval()

with torch.no_grad():

    sample_pred = model(xb)

print("\nInput shape :", tuple(xb.shape))
print("Output shape:", tuple(sample_pred.shape))

assert sample_pred.shape == (
    xb.shape[0],
    NUM_SENSORS
)

assert torch.isfinite(sample_pred).all()

print("\nCELL 5 COMPLETED SUCCESSFULLY")


## Training, validation and checkpointing


In [ ]:

# ============================================================
# CELL 6 — TRAINING, VALIDATION AND CHECKPOINTING
# ============================================================

import time
import copy
import json

# ------------------------------------------------------------
# Optimizer and loss
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

loss_fn = nn.MSELoss()

best_val = float("inf")
best_epoch = 0
patience_counter = 0

train_losses = []
val_losses = []

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

print("\n==========================================")
print("STARTING ST-HopGNN TRAINING")
print("==========================================")

print("Maximum epochs :", EPOCHS)
print("Patience       :", PATIENCE)
print("Learning rate  :", LR)
print("Freeze epoch   :", FREEZE_EPOCH)
print("Checkpoint     :", BEST_MODEL_PATH)

train_start = time.time()

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # Freeze adjacency after 20 completed epochs
    # --------------------------------------------------------

    if epoch == FREEZE_EPOCH:

        model.freeze_adj()

        print(
            "\n=========================================="
        )
        print("ADJACENCY FROZEN AFTER 20 EPOCHS")
        print(
            "==========================================\n"
        )

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        pred = model(xb)

        loss = loss_fn(pred, yb)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            pred = model(xb)

            loss = loss_fn(pred, yb)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    train_losses.append(float(train_loss))
    val_losses.append(float(val_loss))

    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    print(
        f"Epoch {epoch+1:03d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

    # --------------------------------------------------------
    # SAVE BEST VALIDATION MODEL
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss
        best_epoch = epoch + 1

        patience_counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),

                "scaler": train_ds.scaler,

                "epoch": best_epoch,

                "best_val_loss": best_val,

                "config": {
                    "num_sensors": NUM_SENSORS,
                    "window": WINDOW,
                    "horizon": HORIZON,
                    "hidden_dim": HID,
                    "hops": HOPS,
                    "adj_threshold": ADJ_THRESHOLD,
                    "freeze_epoch": FREEZE_EPOCH
                }

            },
            BEST_MODEL_PATH
        )

        print("  Best model saved")

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("\nEarly stopping triggered")
            break

# ------------------------------------------------------------
# Training completed
# ------------------------------------------------------------

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

train_time = time.time() - train_start

epochs_completed = epoch + 1

# ------------------------------------------------------------
# SAVE FINAL TRAINING STATE
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epochs_completed,
        "best_epoch": best_epoch,
        "best_val_loss": best_val,
        "adjacency_frozen": not model.adj.requires_grad
    },
    FINAL_MODEL_PATH
)

# ------------------------------------------------------------
# SAVE TRAINING HISTORY
# ------------------------------------------------------------

history = {
    "train_loss": train_losses,
    "val_loss": val_losses,
    "best_epoch": best_epoch,
    "best_val_loss": float(best_val),
    "epochs_completed": epochs_completed,
    "training_time_seconds": float(train_time)
}

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=4)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n==========================================")
print("TRAINING COMPLETED")
print("==========================================")

print("Epochs completed :", epochs_completed)
print("Best epoch       :", best_epoch)
print("Best val loss    :", f"{best_val:.6f}")
print("Training time(s) :", f"{train_time:.2f}")
print("Adjacency frozen :", not model.adj.requires_grad)

print("\nBest checkpoint :", BEST_MODEL_PATH)
print("Final checkpoint:", FINAL_MODEL_PATH)
print("Training history:", HISTORY_PATH)

print("\nCELL 6 COMPLETED SUCCESSFULLY")


## ST-HopGNN evaluation


In [ ]:

# ============================================================
# CELL 7 — BEST CHECKPOINT AND FINAL LTE EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import torch
import time
import os
import inspect

# ------------------------------------------------------------
# Load best checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Best validation MSE  :", checkpoint["best_val_loss"])

# ------------------------------------------------------------
# Collect normalized predictions and targets
# ------------------------------------------------------------

all_predictions = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        pred = model(xb)

        all_predictions.append(pred.cpu().numpy())
        all_targets.append(yb.cpu().numpy())

y_pred_norm = np.concatenate(all_predictions, axis=0)
y_true_norm = np.concatenate(all_targets, axis=0)

assert y_pred_norm.shape == y_true_norm.shape
assert y_pred_norm.shape[1] == NUM_SENSORS

# ------------------------------------------------------------
# Inverse scaling
# ------------------------------------------------------------

scaler = checkpoint["scaler"]

y_pred = scaler.inverse_transform(y_pred_norm)
y_true = scaler.inverse_transform(y_true_norm)

# ------------------------------------------------------------
# Real-scale evaluation
# ------------------------------------------------------------

errors = y_pred - y_true

mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))

sensor_mae = np.mean(np.abs(errors), axis=0)
sensor_rmse = np.sqrt(np.mean(errors ** 2, axis=0))

print("\n==========================================")
print("FINAL LTE TEST RESULTS — REAL SCALE")
print("==========================================")

print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")

print("Test samples:", len(y_true))

# ------------------------------------------------------------
# Per-sensor metrics
# ------------------------------------------------------------

sensor_results = pd.DataFrame({
    "Sensor": np.arange(1, NUM_SENSORS + 1),
    "MAE": sensor_mae,
    "RMSE": sensor_rmse
})

print("\nPER-SENSOR RESULTS")
print(sensor_results.to_string(index=False))

sensor_results.to_csv(
    os.path.join(
        os.path.dirname(BEST_MODEL_PATH),
        "per_sensor_metrics.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Inference latency
# ------------------------------------------------------------

model.eval()

xb, _ = next(iter(test_loader))
xb = xb[:1].to(DEVICE)

with torch.no_grad():

    for _ in range(30):
        _ = model(xb)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    repetitions = 200

    start = time.perf_counter()

    for _ in range(repetitions):
        _ = model(xb)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

latency_ms = elapsed / repetitions * 1000

print(f"\nInference latency: {latency_ms:.6f} ms/sample")

# ------------------------------------------------------------
# Learned adjacency export
# ------------------------------------------------------------

adj = model.get_adjacency().cpu().numpy()

np.save(
    os.path.join(
        os.path.dirname(BEST_MODEL_PATH),
        "learned_adjacency.npy"
    ),
    adj
)

pd.DataFrame(adj).to_csv(
    os.path.join(
        os.path.dirname(BEST_MODEL_PATH),
        "learned_adjacency.csv"
    ),
    index=False
)

num_edges = np.count_nonzero(adj)
density = num_edges / adj.size

print("\n==========================================")
print("LEARNED ADJACENCY")
print("==========================================")

print("Adjacency shape :", adj.shape)
print("Nonzero edges   :", num_edges)
print("Graph density   :", f"{density:.4f}")

# ------------------------------------------------------------
# Save final metrics
# ------------------------------------------------------------

metrics = {
    "best_epoch": int(checkpoint["epoch"]),
    "best_val_mse": float(checkpoint["best_val_loss"]),
    "test_mae": float(mae),
    "test_rmse": float(rmse),
    "latency_ms_per_sample": float(latency_ms),
    "num_parameters": int(
        sum(p.numel() for p in model.parameters())
    ),
    "nonzero_edges": int(num_edges),
    "graph_density": float(density)
}

import json

with open(
    os.path.join(
        os.path.dirname(BEST_MODEL_PATH),
        "test_metrics.json"
    ),
    "w"
) as f:
    json.dump(metrics, f, indent=4)

print("\nCELL 7 COMPLETED SUCCESSFULLY")


## LSTM baseline


In [ ]:
# ============================================================
# LSTM BASELINE — LTE
# ============================================================

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# CONFIG
# ============================================================

HID = 128

EPOCHS = 80
PATIENCE = 10

LR = 1e-3

MODEL_PATH = os.path.join(RUN_DIR, "LSTM_LTE.pth")

# ============================================================
# MODEL
# ============================================================

class LSTMModel(nn.Module):

    def __init__(self,
                 num_sensors,
                 hid=64):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=num_sensors,
            hidden_size=hid,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Sequential(

            nn.Linear(hid, 64),
            nn.ReLU(),

            nn.Linear(64, num_sensors)
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1]

        out = self.fc(out)

        return out
# ============================================================
# MODEL
# ============================================================

model = LSTMModel(NUM_SENSORS, HID).to(device)

# ============================================================
# PARAM COUNT
# ============================================================

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable Parameters:", params)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LR
)

loss_fn = nn.MSELoss()

# ============================================================
# TRAINING
# ============================================================

best_val = np.inf
patience = 0

train_start = time.time()

print("\nStarting LSTM Training...\n")

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = loss_fn(pred, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += loss_fn(
                pred,
                yb
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss

        patience = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("✅ Best model saved")

    else:

        patience += 1

        if patience >= PATIENCE:

            print("\n⛔ Early stopping")

            break
train_time = time.time() - train_start

# ============================================================
# LOAD BEST
# ============================================================

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

# ============================================================
# TESTING
# ============================================================

preds = []
trues = []

infer_start = time.time()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        pred = model(xb)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            yb.numpy()
        )

infer_time = time.time() - infer_start

preds = np.vstack(preds)
trues = np.vstack(trues)

# ============================================================
# INVERSE SCALE
# ============================================================

preds_real = train_ds.scaler.inverse_transform(preds)

trues_real = train_ds.scaler.inverse_transform(trues)

# ============================================================
# METRICS
# ============================================================

mae = np.mean(
    np.abs(preds_real - trues_real)
)

rmse = np.sqrt(
    np.mean(
        (preds_real - trues_real)**2
    )
)

latency_ms = (
    infer_time / len(test_ds)
) * 1000

# ============================================================
# RESULTS
# ============================================================

print("\n================================================")
print("LSTM LTE RESULTS")
print("================================================")

print(f"MAE              : {mae:.6f}")
print(f"RMSE             : {rmse:.6f}")

print(f"\nParameters       : {params}")

print(f"Training Time(s) : {train_time:.2f}")

print(f"Inference Latency(ms/sample): "
      f"{latency_ms:.4f}")


## TCN baseline


In [ ]:
# ============================================================
# TCN BASELINE — LTE
# ============================================================

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# CONFIG
# ============================================================

HID = 64

EPOCHS = 80
PATIENCE = 10

LR = 1e-3

MODEL_PATH = os.path.join(RUN_DIR, "TCN_LTE.pth")

# ============================================================
# TEMPORAL BLOCK
# ============================================================

class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size,
        dilation,
        dropout=0.3
    ):

        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu1 = nn.ReLU()

        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_ch,
            out_ch,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu2 = nn.ReLU()

        self.drop2 = nn.Dropout(dropout)

        self.downsample = (
            nn.Conv1d(in_ch, out_ch, 1)
            if in_ch != out_ch
            else None
        )

    def forward(self, x):

        out = self.conv1(x)

        out = out[:, :, :-self.conv1.padding[0]]

        out = self.relu1(out)

        out = self.drop1(out)

        out = self.conv2(out)

        out = out[:, :, :-self.conv2.padding[0]]

        out = self.relu2(out)

        out = self.drop2(out)

        res = x if self.downsample is None \
              else self.downsample(x)

        return out + res

# ============================================================
# TCN MODEL
# ============================================================

class TCNModel(nn.Module):

    def __init__(
        self,
        num_sensors,
        hid=64
    ):

        super().__init__()

        self.network = nn.Sequential(

            TemporalBlock(
                num_sensors,
                hid,
                kernel_size=2,
                dilation=1
            ),

            TemporalBlock(
                hid,
                hid,
                kernel_size=2,
                dilation=2
            )
        )

        self.fc = nn.Sequential(

            nn.Linear(hid, 64),

            nn.ReLU(),

            nn.Linear(64, num_sensors)
        )

    def forward(self, x):

        """
        x:
            [B,T,N]
        """

        # --------------------------------------------
        # TCN expects [B,C,T]
        # --------------------------------------------

        x = x.permute(0, 2, 1)

        out = self.network(x)

        # --------------------------------------------
        # LAST TIME STEP
        # --------------------------------------------

        out = out[:, :, -1]

        out = self.fc(out)

        return out

# ============================================================
# MODEL
# ============================================================

model = TCNModel(NUM_SENSORS, HID).to(device)

# ============================================================
# PARAM COUNT
# ============================================================

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable Parameters:", params)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LR
)

loss_fn = nn.MSELoss()

# ============================================================
# TRAINING
# ============================================================

best_val = np.inf
patience = 0

train_start = time.time()

print("\nStarting TCN Training...\n")

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = loss_fn(pred, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += loss_fn(
                pred,
                yb
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss

        patience = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("✅ Best model saved")

    else:

        patience += 1

        if patience >= PATIENCE:

            print("\n⛔ Early stopping")

            break

train_time = time.time() - train_start

# ============================================================
# LOAD BEST
# ============================================================

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

# ============================================================
# TESTING
# ============================================================

preds = []
trues = []

infer_start = time.time()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        pred = model(xb)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            yb.numpy()
        )

infer_time = time.time() - infer_start

preds = np.vstack(preds)
trues = np.vstack(trues)

# ============================================================
# INVERSE SCALE
# ============================================================

preds_real = train_ds.scaler.inverse_transform(preds)

trues_real = train_ds.scaler.inverse_transform(trues)

# ============================================================
# METRICS
# ============================================================

mae = np.mean(
    np.abs(preds_real - trues_real)
)

rmse = np.sqrt(
    np.mean(
        (preds_real - trues_real)**2
    )
)

latency_ms = (
    infer_time / len(test_ds)
) * 1000

# ============================================================
# RESULTS
# ============================================================

print("\n================================================")
print("TCN LTE RESULTS")
print("================================================")

print(f"MAE              : {mae:.6f}")
print(f"RMSE             : {rmse:.6f}")

print(f"\nParameters       : {params}")

print(f"Training Time(s) : {train_time:.2f}")

print(
    f"Inference Latency(ms/sample): "
    f"{latency_ms:.4f}"
)

## STGCN baseline


In [ ]:
# ============================================================
# STGCN BASELINE — LTE
# ============================================================

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torch_geometric.nn import GCNConv

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# CONFIG
# ============================================================

HID = 64

EPOCHS = 80
PATIENCE = 10

LR = 1e-3

MODEL_PATH = os.path.join(RUN_DIR, "STGCN_LTE.pth")

# ============================================================
# FIXED GRAPH
# ============================================================

adj = torch.ones(NUM_SENSORS, NUM_SENSORS)

edge_index = adj.nonzero().t().contiguous()

edge_weight = adj[
    edge_index[0],
    edge_index[1]
]

edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)

# ============================================================
# TEMPORAL BLOCK
# ============================================================

class TemporalConv(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel=3
    ):

        super().__init__()

        self.conv = nn.Conv1d(
            in_ch,
            out_ch,
            kernel,
            padding=1
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        return self.relu(
            self.conv(x)
        )

# ============================================================
# STGCN BLOCK
# ============================================================

class STGCNBlock(nn.Module):

    def __init__(self, hid):

        super().__init__()

        self.temp1 = TemporalConv(
            hid,
            hid
        )

        self.gcn = GCNConv(
            hid,
            hid
        )

        self.temp2 = TemporalConv(
            hid,
            hid
        )

        self.drop = nn.Dropout(0.3)

    def forward(self, x):

        """
        x:
            [B,T,N,H]
        """

        B, T, N, H = x.shape

        # ----------------------------------------------------
        # TEMPORAL
        # ----------------------------------------------------

        x = x.permute(0,2,3,1)

        x = x.reshape(B*N, H, T)

        x = self.temp1(x)

        # ----------------------------------------------------
        # LAST TEMPORAL STATE
        # ----------------------------------------------------

        x_last = x[:, :, -1]

        # ----------------------------------------------------
        # GCN
        # ----------------------------------------------------

        x_last = self.gcn(
            x_last,
            edge_index,
            edge_weight
        )

        x_last = self.drop(x_last)

        # ----------------------------------------------------
        # TEMPORAL AGAIN
        # ----------------------------------------------------

        x_last = x_last.view(B, N, H)

        x_last = x_last.permute(0,2,1)

        x_last = self.temp2(x_last)

        x_last = x_last.permute(0,2,1)

        return x_last

# ============================================================
# STGCN MODEL
# ============================================================

class STGCNModel(nn.Module):

    def __init__(
        self,
        num_sensors,
        hid=64
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            1,
            hid
        )

        self.block1 = STGCNBlock(hid)

        self.block2 = STGCNBlock(hid)

        self.fc = nn.Linear(
            hid,
            1
        )

    def forward(self, x):

        """
        x:
            [B,T,N]
        """

        B, T, N = x.shape

        x = x.unsqueeze(-1)

        x = self.input_proj(x)

        x = self.block1(x)

        x = self.block2(
            x.unsqueeze(1)
        )

        out = self.fc(x)

        out = out.squeeze(-1)

        return out

# ============================================================
# MODEL
# ============================================================

model = STGCNModel(
    NUM_SENSORS,
    HID
).to(device)

# ============================================================
# PARAM COUNT
# ============================================================

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable Parameters:", params)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LR
)

loss_fn = nn.MSELoss()

# ============================================================
# TRAINING
# ============================================================

best_val = np.inf
patience = 0

train_start = time.time()

print("\nStarting STGCN Training...\n")

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = loss_fn(pred, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += loss_fn(
                pred,
                yb
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss

        patience = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("✅ Best model saved")

    else:

        patience += 1

        if patience >= PATIENCE:

            print("\n⛔ Early stopping")

            break

train_time = time.time() - train_start

# ============================================================
# LOAD BEST
# ============================================================

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

# ============================================================
# TESTING
# ============================================================

preds = []
trues = []

infer_start = time.time()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        pred = model(xb)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            yb.numpy()
        )

infer_time = time.time() - infer_start

preds = np.vstack(preds)
trues = np.vstack(trues)

# ============================================================
# INVERSE SCALE
# ============================================================

preds_real = train_ds.scaler.inverse_transform(preds)

trues_real = train_ds.scaler.inverse_transform(trues)

# ============================================================
# METRICS
# ============================================================

mae = np.mean(
    np.abs(preds_real - trues_real)
)

rmse = np.sqrt(
    np.mean(
        (preds_real - trues_real)**2
    )
)

latency_ms = (
    infer_time / len(test_ds)
) * 1000

# ============================================================
# RESULTS
# ============================================================

print("\n================================================")
print("STGCN LTE RESULTS")
print("================================================")

print(f"MAE              : {mae:.6f}")
print(f"RMSE             : {rmse:.6f}")

print(f"\nParameters       : {params}")

print(f"Training Time(s) : {train_time:.2f}")

print(
    f"Inference Latency(ms/sample): "
    f"{latency_ms:.4f}"
)

## AGCRN baseline


In [ ]:
# ============================================================
# AGCRN BASELINE — LTE
# Adaptive Graph Convolutional Recurrent Network
# ============================================================

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# CONFIG
# ============================================================

HID = 64
EMB = 16

EPOCHS = 80
PATIENCE = 10

LR = 1e-3

MODEL_PATH = os.path.join(RUN_DIR, "AGCRN_LTE.pth")

# ============================================================
# ADAPTIVE GRAPH CONV
# ============================================================

class AVWGCN(nn.Module):

    def __init__(
        self,
        dim_in,
        dim_out,
        cheb_k,
        embed_dim
    ):

        super().__init__()

        self.cheb_k = cheb_k

        self.weights_pool = nn.Parameter(
            torch.FloatTensor(
                embed_dim,
                cheb_k,
                dim_in,
                dim_out
            )
        )

        self.bias_pool = nn.Parameter(
            torch.FloatTensor(
                embed_dim,
                dim_out
            )
        )

        nn.init.xavier_normal_(
            self.weights_pool
        )

        nn.init.xavier_normal_(
            self.bias_pool
        )

    def forward(
        self,
        x,
        node_emb
    ):

        """
        x:
            [B,N,C]
        """

        N = node_emb.shape[0]

        # ----------------------------------------------------
        # ADAPTIVE ADJ
        # ----------------------------------------------------

        supports = torch.softmax(
            torch.relu(
                torch.mm(
                    node_emb,
                    node_emb.t()
                )
            ),
            dim=1
        )

        support_set = [
            torch.eye(
                N,
                device=x.device
            ),
            supports
        ]

        # ----------------------------------------------------
        # CHEB SUPPORTS
        # ----------------------------------------------------

        for k in range(2, self.cheb_k):

            support_set.append(
                torch.matmul(
                    2 * supports,
                    support_set[-1]
                ) - support_set[-2]
            )

        supports = torch.stack(
            support_set,
            dim=0
        )

        # ----------------------------------------------------
        # GRAPH CONV
        # ----------------------------------------------------

        weights = torch.einsum(
            'nd,dkio->nkio',
            node_emb,
            self.weights_pool
        )

        bias = torch.matmul(
            node_emb,
            self.bias_pool
        )

        x_g = torch.einsum(
            "knm,bmc->bknc",
            supports,
            x
        )

        x_g = x_g.permute(
            0,
            2,
            1,
            3
        )

        out = torch.einsum(
            'bnki,nkio->bno',
            x_g,
            weights
        ) + bias

        return out

# ============================================================
# AGCRN CELL
# ============================================================

class AGCRNCell(nn.Module):

    def __init__(
        self,
        num_nodes,
        dim_in,
        dim_out,
        cheb_k,
        embed_dim
    ):

        super().__init__()

        self.hidden_dim = dim_out

        self.gate = AVWGCN(
            dim_in + dim_out,
            2 * dim_out,
            cheb_k,
            embed_dim
        )

        self.update = AVWGCN(
            dim_in + dim_out,
            dim_out,
            cheb_k,
            embed_dim
        )

    def forward(
        self,
        x,
        state,
        node_emb
    ):

        """
        x:
            [B,N,C]
        """

        state = state.to(x.device)

        input_and_state = torch.cat(
            [x, state],
            dim=-1
        )

        z_r = torch.sigmoid(
            self.gate(
                input_and_state,
                node_emb
            )
        )

        z, r = torch.split(
            z_r,
            self.hidden_dim,
            dim=-1
        )

        candidate = torch.cat(
            [x, z * state],
            dim=-1
        )

        hc = torch.tanh(
            self.update(
                candidate,
                node_emb
            )
        )

        h = r * state + (1-r) * hc

        return h

# ============================================================
# AGCRN MODEL
# ============================================================

class AGCRN(nn.Module):

    def __init__(
        self,
        num_nodes,
        input_dim=1,
        hidden_dim=64,
        output_dim=1,
        cheb_k=2,
        embed_dim=16
    ):

        super().__init__()

        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim

        self.node_emb = nn.Parameter(
            torch.randn(
                num_nodes,
                embed_dim
            )
        )

        self.cell = AGCRNCell(
            num_nodes,
            input_dim,
            hidden_dim,
            cheb_k,
            embed_dim
        )

        self.proj = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        """
        x:
            [B,T,N]
        """

        B, T, N = x.shape

        h = torch.zeros(
            B,
            N,
            self.hidden_dim,
            device=x.device
        )

        # ----------------------------------------------------
        # RECURRENT PROCESSING
        # ----------------------------------------------------

        for t in range(T):

            xt = x[:, t].unsqueeze(-1)

            h = self.cell(
                xt,
                h,
                self.node_emb
            )

        out = self.proj(h)

        out = out.squeeze(-1)

        return out

# ============================================================
# MODEL
# ============================================================

model = AGCRN(
    NUM_SENSORS,
    hidden_dim=HID,
    embed_dim=EMB
).to(device)

# ============================================================
# PARAM COUNT
# ============================================================

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable Parameters:", params)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LR
)

loss_fn = nn.MSELoss()

# ============================================================
# TRAINING
# ============================================================

best_val = np.inf
patience = 0

train_start = time.time()

print("\nStarting AGCRN Training...\n")

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = loss_fn(pred, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += loss_fn(
                pred,
                yb
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss

        patience = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("✅ Best model saved")

    else:

        patience += 1

        if patience >= PATIENCE:

            print("\n⛔ Early stopping")

            break

train_time = time.time() - train_start

# ============================================================
# LOAD BEST
# ============================================================

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

# ============================================================
# TESTING
# ============================================================

preds = []
trues = []

infer_start = time.time()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        pred = model(xb)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            yb.numpy()
        )

infer_time = time.time() - infer_start

preds = np.vstack(preds)
trues = np.vstack(trues)

# ============================================================
# INVERSE SCALE
# ============================================================

preds_real = train_ds.scaler.inverse_transform(preds)

trues_real = train_ds.scaler.inverse_transform(trues)

# ============================================================
# METRICS
# ============================================================

mae = np.mean(
    np.abs(preds_real - trues_real)
)

rmse = np.sqrt(
    np.mean(
        (preds_real - trues_real)**2
    )
)

latency_ms = (
    infer_time / len(test_ds)
) * 1000

# ============================================================
# RESULTS
# ============================================================

print("\n================================================")
print("AGCRN LTE RESULTS")
print("================================================")

print(f"MAE              : {mae:.6f}")
print(f"RMSE             : {rmse:.6f}")

print(f"\nParameters       : {params}")

print(f"Training Time(s) : {train_time:.2f}")

print(
    f"Inference Latency(ms/sample): "
    f"{latency_ms:.4f}"
)

## DCRNN baseline


In [ ]:
# ============================================================
# DCRNN BASELINE — LTE
# Diffusion Convolutional Recurrent Neural Network
# ============================================================

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# CONFIG
# ============================================================

HID = 64
K = 2

EPOCHS = 80
PATIENCE = 10

LR = 1e-3

MODEL_PATH = os.path.join(RUN_DIR, "DCRNN_LTE.pth")

# ============================================================
# FIXED ADJACENCY
# ============================================================

adj = torch.ones(NUM_SENSORS, NUM_SENSORS)

# row-normalized adjacency
adj = adj / adj.sum(dim=1, keepdim=True)

adj = adj.to(device)

# ============================================================
# DIFFUSION CONV
# ============================================================

class DiffusionConv(nn.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        k=2
    ):

        super().__init__()

        self.k = k

        self.theta = nn.Linear(
            in_dim * k,
            out_dim
        )

    def forward(self, x, adj):

        """
        x:
            [B,N,C]
        """

        supports = []

        x0 = x

        supports.append(x0)

        x1 = torch.einsum(
            "nm,bmc->bnc",
            adj,
            x0
        )

        supports.append(x1)

        for _ in range(2, self.k):

            x2 = 2 * torch.einsum(
                "nm,bmc->bnc",
                adj,
                x1
            ) - x0

            supports.append(x2)

            x0, x1 = x1, x2

        out = torch.cat(
            supports,
            dim=-1
        )

        out = self.theta(out)

        return out

# ============================================================
# DCRNN CELL
# ============================================================

class DCRNNCell(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        k=2
    ):

        super().__init__()

        self.hidden_dim = hidden_dim

        self.gate = DiffusionConv(
            input_dim + hidden_dim,
            2 * hidden_dim,
            k
        )

        self.update = DiffusionConv(
            input_dim + hidden_dim,
            hidden_dim,
            k
        )

    def forward(
        self,
        x,
        h,
        adj
    ):

        """
        x:
            [B,N,C]
        """

        combined = torch.cat(
            [x, h],
            dim=-1
        )

        z_r = torch.sigmoid(
            self.gate(
                combined,
                adj
            )
        )

        z, r = torch.split(
            z_r,
            self.hidden_dim,
            dim=-1
        )

        candidate = torch.cat(
            [x, z * h],
            dim=-1
        )

        hc = torch.tanh(
            self.update(
                candidate,
                adj
            )
        )

        h = r * h + (1-r) * hc

        return h

# ============================================================
# DCRNN MODEL
# ============================================================

class DCRNN(nn.Module):

    def __init__(
        self,
        num_nodes,
        input_dim=1,
        hidden_dim=64,
        output_dim=1,
        k=2
    ):

        super().__init__()

        self.hidden_dim = hidden_dim

        self.cell = DCRNNCell(
            input_dim,
            hidden_dim,
            k
        )

        self.proj = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        """
        x:
            [B,T,N]
        """

        B, T, N = x.shape

        h = torch.zeros(
            B,
            N,
            self.hidden_dim,
            device=x.device
        )

        # ----------------------------------------------------
        # RECURRENT DIFFUSION
        # ----------------------------------------------------

        for t in range(T):

            xt = x[:, t].unsqueeze(-1)

            h = self.cell(
                xt,
                h,
                adj
            )

        out = self.proj(h)

        out = out.squeeze(-1)

        return out

# ============================================================
# MODEL
# ============================================================

model = DCRNN(
    NUM_SENSORS,
    hidden_dim=HID,
    k=K
).to(device)

# ============================================================
# PARAM COUNT
# ============================================================

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable Parameters:", params)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LR
)

loss_fn = nn.MSELoss()

# ============================================================
# TRAINING
# ============================================================

best_val = np.inf
patience = 0

train_start = time.time()

print("\nStarting DCRNN Training...\n")

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = loss_fn(pred, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += loss_fn(
                pred,
                yb
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss

        patience = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("✅ Best model saved")

    else:

        patience += 1

        if patience >= PATIENCE:

            print("\n⛔ Early stopping")

            break

train_time = time.time() - train_start

# ============================================================
# LOAD BEST
# ============================================================

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

# ============================================================
# TESTING
# ============================================================

preds = []
trues = []

infer_start = time.time()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        pred = model(xb)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            yb.numpy()
        )

infer_time = time.time() - infer_start

preds = np.vstack(preds)
trues = np.vstack(trues)

# ============================================================
# INVERSE SCALE
# ============================================================

preds_real = train_ds.scaler.inverse_transform(preds)

trues_real = train_ds.scaler.inverse_transform(trues)

# ============================================================
# METRICS
# ============================================================

mae = np.mean(
    np.abs(preds_real - trues_real)
)

rmse = np.sqrt(
    np.mean(
        (preds_real - trues_real)**2
    )
)

latency_ms = (
    infer_time / len(test_ds)
) * 1000

# ============================================================
# RESULTS
# ============================================================

print("\n================================================")
print("DCRNN LTE RESULTS")
print("================================================")

print(f"MAE              : {mae:.6f}")
print(f"RMSE             : {rmse:.6f}")

print(f"\nParameters       : {params}")

print(f"Training Time(s) : {train_time:.2f}")

print(
    f"Inference Latency(ms/sample): "
    f"{latency_ms:.4f}"
)

## MTGNN baseline


In [ ]:
# ============================================================
# MTGNN BASELINE — LTE
# Multivariate Time-Series Graph Neural Network
# Simplified MTGNN-style implementation
# ============================================================

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# CONFIG
# ============================================================

HID = 64
NODE_EMB = 16
TOPK = 8

EPOCHS = 80
PATIENCE = 10

LR = 1e-3

MODEL_PATH = os.path.join(RUN_DIR, "MTGNN_LTE.pth")

# ============================================================
# GRAPH LEARNING
# ============================================================

class GraphLearning(nn.Module):

    def __init__(
        self,
        num_nodes,
        emb_dim=16,
        topk=8
    ):

        super().__init__()

        self.emb1 = nn.Parameter(
            torch.randn(num_nodes, emb_dim)
        )

        self.emb2 = nn.Parameter(
            torch.randn(num_nodes, emb_dim)
        )

        self.topk = topk

    def forward(self):

        # ----------------------------------------------------
        # LEARN ADJ
        # ----------------------------------------------------

        adj = torch.relu(
            torch.mm(
                self.emb1,
                self.emb2.t()
            )
        )

        # ----------------------------------------------------
        # TOP-K SPARSIFICATION
        # ----------------------------------------------------

        mask = torch.zeros_like(adj)

        vals, idx = torch.topk(
            adj,
            self.topk,
            dim=1
        )

        mask.scatter_(1, idx, 1.0)

        adj = adj * mask

        # ----------------------------------------------------
        # NORMALIZE
        # ----------------------------------------------------

        adj = adj / (
            adj.sum(dim=1, keepdim=True) + 1e-6
        )

        return adj

# ============================================================
# MIXHOP GRAPH CONV
# ============================================================

class MixHopLayer(nn.Module):

    def __init__(
        self,
        in_dim,
        out_dim
    ):

        super().__init__()

        self.lin1 = nn.Linear(
            in_dim,
            out_dim
        )

        self.lin2 = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(
        self,
        x,
        adj
    ):

        """
        x:
            [B,N,C]
        """

        hop1 = torch.einsum(
            "nm,bmc->bnc",
            adj,
            x
        )

        hop2 = torch.einsum(
            "nm,bmc->bnc",
            adj,
            hop1
        )

        out = self.lin1(hop1) + \
              self.lin2(hop2)

        return out

# ============================================================
# TEMPORAL CONV
# ============================================================

class TemporalConv(nn.Module):

    def __init__(
        self,
        channels
    ):

        super().__init__()

        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        return self.relu(
            self.conv(x)
        )

# ============================================================
# MTGNN MODEL
# ============================================================

class MTGNN(nn.Module):

    def __init__(
        self,
        num_nodes,
        hid=64
    ):

        super().__init__()

        self.graph_learner = GraphLearning(
            num_nodes,
            emb_dim=NODE_EMB,
            topk=TOPK
        )

        self.input_proj = nn.Linear(
            1,
            hid
        )

        self.temp1 = TemporalConv(hid)

        self.graph1 = MixHopLayer(
            hid,
            hid
        )

        self.temp2 = TemporalConv(hid)

        self.graph2 = MixHopLayer(
            hid,
            hid
        )

        self.drop = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hid,
            1
        )

    def forward(self, x):

        """
        x:
            [B,T,N]
        """

        B, T, N = x.shape

        # ----------------------------------------------------
        # LEARN GRAPH
        # ----------------------------------------------------

        adj = self.graph_learner()

        # ----------------------------------------------------
        # INPUT PROJ
        # ----------------------------------------------------

        x = x.unsqueeze(-1)

        x = self.input_proj(x)

        # ----------------------------------------------------
        # TEMPORAL
        # ----------------------------------------------------

        x = x.permute(0,2,3,1)

        x = x.reshape(B*N, -1, T)

        x = self.temp1(x)

        x = x[:, :, -1]

        x = x.view(B, N, -1)

        # ----------------------------------------------------
        # GRAPH
        # ----------------------------------------------------

        x = self.graph1(x, adj)

        x = self.drop(x)

        # ----------------------------------------------------
        # SECOND GRAPH
        # ----------------------------------------------------

        x = self.graph2(x, adj)

        x = self.drop(x)

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        out = self.fc(x)

        out = out.squeeze(-1)

        return out

# ============================================================
# MODEL
# ============================================================

model = MTGNN(
    NUM_SENSORS,
    hid=HID
).to(device)

# ============================================================
# PARAM COUNT
# ============================================================

params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable Parameters:", params)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LR
)

loss_fn = nn.MSELoss()

# ============================================================
# TRAINING
# ============================================================

best_val = np.inf
patience = 0

train_start = time.time()

print("\nStarting MTGNN Training...\n")

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = loss_fn(pred, yb)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)

            val_loss += loss_fn(
                pred,
                yb
            ).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    # --------------------------------------------------------
    # SAVE BEST
    # --------------------------------------------------------

    if val_loss < best_val:

        best_val = val_loss

        patience = 0

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("✅ Best model saved")

    else:

        patience += 1

        if patience >= PATIENCE:

            print("\n⛔ Early stopping")

            break

train_time = time.time() - train_start

# ============================================================
# LOAD BEST
# ============================================================

model.load_state_dict(
    torch.load(MODEL_PATH)
)

model.eval()

# ============================================================
# TESTING
# ============================================================

preds = []
trues = []

infer_start = time.time()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        pred = model(xb)

        preds.append(
            pred.cpu().numpy()
        )

        trues.append(
            yb.numpy()
        )

infer_time = time.time() - infer_start

preds = np.vstack(preds)
trues = np.vstack(trues)

# ============================================================
# INVERSE SCALE
# ============================================================

preds_real = train_ds.scaler.inverse_transform(preds)

trues_real = train_ds.scaler.inverse_transform(trues)

# ============================================================
# METRICS
# ============================================================

mae = np.mean(
    np.abs(preds_real - trues_real)
)

rmse = np.sqrt(
    np.mean(
        (preds_real - trues_real)**2
    )
)

latency_ms = (
    infer_time / len(test_ds)
) * 1000

# ============================================================
# RESULTS
# ============================================================

print("\n================================================")
print("MTGNN LTE RESULTS")
print("================================================")

print(f"MAE              : {mae:.6f}")
print(f"RMSE             : {rmse:.6f}")

print(f"\nParameters       : {params}")

print(f"Training Time(s) : {train_time:.2f}")

print(
    f"Inference Latency(ms/sample): "
    f"{latency_ms:.4f}"
)